In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins_4km2.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [6]:
from data import BasinDataLake

datalake = BasinDataLake(save_dir / 'datalakes' / 'training')
# datalake.optimize(0)

In [8]:
import xarray as xr

sub_counts = subs_df['outlet_id'].value_counts().to_dict()
subs_df = subs_df.sort_values(by='outlet_id', key = lambda x: x.map(sub_counts), ascending=False)

BATCH_SIZE = 1000
batches = []
for basin_id, basin_group in subs_df.groupby('outlet_id'):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        batches.append((basin_id, batch_subs))

def get_batch_era5_df(basin_id, batch_subs):
    root = save_dir / f"datalakes/training/dynamic_data/era5/basin={basin_id}"

    dfs = []
    for d in root.iterdir():
        tmp_df = pd.concat([pd.read_parquet(f) for f in d.glob("*.parquet")])
        tmp_df = tmp_df.set_index(['subbasin','date'])
        tmp_df = tmp_df.loc[pd.IndexSlice[batch_subs, :], :]
        dfs.append(tmp_df)
    
    df = pd.concat(dfs)
    df = df.loc[~df.index.duplicated(keep='first'), :]
    return df

processed_basins = set()
failed_basins = {}
for basin_id, batch_subs in tqdm(batches):
    basin_path = store.store_path / basin_id 
    if (basin_path / 'ro_mean').is_dir():
        ds = xr.open_zarr(basin_path)
        nan_count = ds['ro_mean'].isnull().sum().compute().item()
        if nan_count == 0:
            continue
    
    basin_subs = basin_subbasin_dict[basin_id]
    batch_df_list = []

    try:
        batch_df = get_batch_era5_df(basin_id, batch_subs)
        init_vars = basin_id not in processed_basins
        store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs, init_vars)
        processed_basins.union({basin_id})
    except Exception as e: 
        failed_basins[basin_id] = e
        raise e

100%|██████████| 3/3 [05:28<00:00, 109.37s/it]


In [6]:
batch_df

ssrd_mean     strd_mean  \
subbasin       date                                                    
56155034       2015-01-01 00:00:00+00:00  3.176756e+07  2.818138e+07   
               2015-01-02 00:00:00+00:00  3.223902e+07  2.863344e+07   
               2015-01-03 00:00:00+00:00  2.381179e+07  2.815478e+07   
               2015-01-04 00:00:00+00:00  3.275706e+07  2.696731e+07   
               2015-01-05 00:00:00+00:00  3.288375e+07  2.992960e+07   
...                                                ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  1.834709e+07  2.877304e+07   
               2025-12-28 00:00:00+00:00  3.235656e+07  2.654268e+07   
               2025-12-29 00:00:00+00:00  3.310289e+07  2.744607e+07   
               2025-12-30 00:00:00+00:00  3.296509e+07  2.709251e+07   
               2025-12-31 00:00:00+00:00  3.309605e+07  2.732326e+07   

                                               ro_mean      sro_mean  \
subbasin       date                                                    
56155034       2015-01-01 00:00:00+00:00  1.503372e-08  0.000000e+00   
               2015-01-02 00:00:00+00:00  1.722280e-08  0.000000e+00   
               2015-01-03 00:00:00+00:00  2.980232e-08  0.000000e+00   
               2015-01-04 00:00:00+00:00  2.980232e-08  0.000000e+00   
               2015-01-05 00:00:00+00:00  2.246098e-08  0.000000e+00   
...                                                ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  2.607327e-06  4.193518e-08   
               2025-12-28 00:00:00+00:00  2.535080e-06  0.000000e+00   
               2025-12-29 00:00:00+00:00  2.517255e-06  0.000000e+00   
               2025-12-30 00:00:00+00:00  2.469628e-06  0.000000e+00   
               2025-12-31 00:00:00+00:00  2.456665e-06  2.285194e-09   

                                            e_mean       tp_mean  \
subbasin       date                                                
56155034       2015-01-01 00:00:00+00:00 -0.000341  1.290134e-06   
               2015-01-02 00:00:00+00:00 -0.000444  2.145250e-06   
               2015-01-03 00:00:00+00:00 -0.000251  1.299381e-06   
               2015-01-04 00:00:00+00:00 -0.000308  2.990087e-06   
               2015-01-05 00:00:00+00:00 -0.000424  8.574305e-07   
...                                            ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00 -0.000977  2.342487e-05   
               2025-12-28 00:00:00+00:00 -0.001240  2.131183e-06   
               2025-12-29 00:00:00+00:00 -0.001236  1.284480e-06   
               2025-12-30 00:00:00+00:00 -0.001250  8.523463e-07   
               2025-12-31 00:00:00+00:00 -0.001217  2.137012e-06   

                                               sp_mean    d2m_mean  \
subbasin       date                                                  
56155034       2015-01-01 00:00:00+00:00  97239.430646  281.680918   
               2015-01-02 00:00:00+00:00  97409.321352  282.850554   
               2015-01-03 00:00:00+00:00  97787.337662  278.965451   
               2015-01-04 00:00:00+00:00  97599.654488  282.527995   
               2015-01-05 00:00:00+00:00  97239.071113  282.829329   
...                                                ...         ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  98453.178023  278.708177   
               2025-12-28 00:00:00+00:00  98552.394148  281.244677   
               2025-12-29 00:00:00+00:00  98813.668509  280.405255   
               2025-12-30 00:00:00+00:00  98675.470554  280.409691   
               2025-12-31 00:00:00+00:00  98134.370378  279.553260   

                                            t2m_mean  fal_mean      ssrd_var  \
subbasin       date                                                            
56155034       2015-01-01 00:00:00+00:00  295.512250  0.210996  2.469542e+10   
               2015-01-02 00:00:00+00:00  292.707864  0.211212  1.562200e+09   
               2015-01-03 00:00:00+00:00  292.915523  0

In [11]:
import xarray as xr
basin_path = store.store_path / basin_id
ds = xr.open_zarr(basin_path)

ds

<xarray.Dataset> Size: 61MB
Dimensions:         (date: 16437, subbasin: 20)
Coordinates:
  * date            (date) datetime64[ns] 131kB 1980-01-01 ... 2024-12-31
  * subbasin        (subbasin) <U14 1kB '56155034' ... 'ABOM-102984010'
Data variables: (12/23)
    d2m_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    d2m_var         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    discharge       (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    e_mean          (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    e_var           (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    fal_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    ...              ...
    strd_var        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    t2m_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    t2m_var         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    tp_mean         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    tp_var          (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    unit_discharge  (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
Attributes:
    normalization_stats:  {'d2m_mean': {'count': 328740, 'sum': 92463169.5251...